In [ ]:
# doesn't work right now
# block 2
print("="*70)
print("BLOCK 2: FIXED POINT VS NONLINEAR ANALYSIS")
print("="*70)

DERIVATIVE_THRESHOLD = 0.01
NONLINEAR_THRESHOLD = -0.01

fixedpt_mean_heatmap = np.zeros((num_q, num_p))
fixedpt_std_heatmap = np.zeros((num_q, num_p))
nonlinear_mean_heatmap = np.zeros((num_q, num_p))
nonlinear_std_heatmap = np.zeros((num_q, num_p))

print(f"Analyzing fixed point vs nonlinear neurons...")
print(f"  Fixed point threshold: |dx/dt| < {DERIVATIVE_THRESHOLD}")
print(f"  Nonlinear threshold: Wx+1 < {NONLINEAR_THRESHOLD}\n")

for p_idx in range(num_p):
    for q_idx in range(num_q):
        key = (ANALYZE_MATRIX_SIZE, p_idx, q_idx)
        
        if key not in all_final_states or key not in all_weight_matrices:
            print(f"Warning: No data for p_idx={p_idx}, q_idx={q_idx}")
            continue
        
        trial_fixedpt_counts = []
        trial_nonlinear_counts = []
        
        for final_state, W in zip(all_final_states[key], all_weight_matrices[key]):
            input_to_relu = W @ final_state + 1
            
            num_nonlinear = np.sum(input_to_relu < NONLINEAR_THRESHOLD)
            trial_nonlinear_counts.append(num_nonlinear)
            
            derivative = -final_state + np.maximum(0, input_to_relu)
            num_fixedpt = np.sum(np.abs(derivative) < DERIVATIVE_THRESHOLD)
            trial_fixedpt_counts.append(num_fixedpt)
        
        fixedpt_mean_heatmap[q_idx, p_idx] = np.mean(trial_fixedpt_counts)
        fixedpt_std_heatmap[q_idx, p_idx] = np.std(trial_fixedpt_counts)
        nonlinear_mean_heatmap[q_idx, p_idx] = np.mean(trial_nonlinear_counts)
        nonlinear_std_heatmap[q_idx, p_idx] = np.std(trial_nonlinear_counts)
        
        p_val = P_VALS[p_idx]
        q_val = Q_VALS[q_idx]
        print(f"p={p_val:.1f}, q={q_val:.1f} | "
              f"FixedPt: {np.mean(trial_fixedpt_counts):.1f}±{np.std(trial_fixedpt_counts):.1f} | "
              f"Nonlinear: {np.mean(trial_nonlinear_counts):.1f}±{np.std(trial_nonlinear_counts):.1f}")

# viz fp vs nonlinear

p_step = P_VALS[1] - P_VALS[0] if len(P_VALS) > 1 else 1
q_step = Q_VALS[1] - Q_VALS[0] if len(Q_VALS) > 1 else 1
extent = [P_VALS[0] - p_step/2, P_VALS[-1] + p_step/2, 
          Q_VALS[0] - q_step/2, Q_VALS[-1] + q_step/2]

# Adaptive color scale
expected_max_active = max(ANALYZE_MATRIX_SIZE * 0.8, 3 * np.log(ANALYZE_MATRIX_SIZE))
actual_max = max(np.max(fixedpt_mean_heatmap), np.max(nonlinear_mean_heatmap))
vmax_count = min(ANALYZE_MATRIX_SIZE, max(expected_max_active, actual_max * 1.1))
vmax_std = vmax_count / 4

# --- Figure 1: Mean counts comparison ---
fig_means, axes_means = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)
fig_means.suptitle(r'\Large Fixed Point vs Nonlinear Neurons (N=' + f'{ANALYZE_MATRIX_SIZE})', 
                   fontsize=14, y=0.97)

# Fixed point neurons
im1 = axes_means[0].imshow(fixedpt_mean_heatmap, origin='lower', 
                            extent=extent, aspect='auto', 
                            cmap='cividis', interpolation='nearest',
                            vmin=0, vmax=vmax_count)
axes_means[0].set_title(r'Fixed Point Neurons ($|dx/dt| < $' + f'{DERIVATIVE_THRESHOLD})',
                        fontsize=10)
axes_means[0].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_means[0].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_means[0].set_xticks(P_VALS)
axes_means[0].set_yticks(Q_VALS)
axes_means[0].tick_params(labelsize=9)
cbar1 = fig_means.colorbar(im1, ax=axes_means[0])
cbar1.set_label(r'Mean count', fontsize=10)
contours1 = axes_means[0].contour(P_VALS, Q_VALS, fixedpt_mean_heatmap,
                                   colors='white', alpha=0.3, linewidths=0.5, levels=5)
axes_means[0].clabel(contours1, inline=True, fontsize=6)

# Nonlinear neurons
im2 = axes_means[1].imshow(nonlinear_mean_heatmap, origin='lower', 
                            extent=extent, aspect='auto', 
                            cmap='plasma', interpolation='nearest',
                            vmin=0, vmax=vmax_count)
axes_means[1].set_title(r'Nonlinear Neurons (ReLU clipping)', fontsize=10)
axes_means[1].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_means[1].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_means[1].set_xticks(P_VALS)
axes_means[1].set_yticks(Q_VALS)
axes_means[1].tick_params(labelsize=9)
cbar2 = fig_means.colorbar(im2, ax=axes_means[1])
cbar2.set_label(r'Mean count', fontsize=10)
contours2 = axes_means[1].contour(P_VALS, Q_VALS, nonlinear_mean_heatmap,
                                   colors='white', alpha=0.3, linewidths=0.5, levels=5)
axes_means[1].clabel(contours2, inline=True, fontsize=6)

filepath_means_pdf = os.path.join(run_folder, 'fixedpt_vs_nonlinear_means.pdf')
filepath_means_pgf = os.path.join(run_folder, 'fixedpt_vs_nonlinear_means.pgf')
fig_means.savefig(filepath_means_pdf, bbox_inches='tight')
fig_means.savefig(filepath_means_pgf, bbox_inches='tight')

# --- Figure 2: Standard deviations ---
fig_stds, axes_stds = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)
fig_stds.suptitle(r'\Large Variability: Fixed Point vs Nonlinear (N=' + f'{ANALYZE_MATRIX_SIZE})', 
                  fontsize=14, y=0.97)

im1s = axes_stds[0].imshow(fixedpt_std_heatmap, origin='lower', 
                            extent=extent, aspect='auto', 
                            cmap='cividis', interpolation='nearest',
                            vmin=0, vmax=vmax_std)
axes_stds[0].set_title(r'Fixed Point Neurons', fontsize=10)
axes_stds[0].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_stds[0].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_stds[0].set_xticks(P_VALS)
axes_stds[0].set_yticks(Q_VALS)
axes_stds[0].tick_params(labelsize=9)
cbar1s = fig_stds.colorbar(im1s, ax=axes_stds[0])
cbar1s.set_label(r'Std dev', fontsize=10)

im2s = axes_stds[1].imshow(nonlinear_std_heatmap, origin='lower', 
                            extent=extent, aspect='auto', 
                            cmap='plasma', interpolation='nearest',
                            vmin=0, vmax=vmax_std)
axes_stds[1].set_title(r'Nonlinear Neurons', fontsize=10)
axes_stds[1].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_stds[1].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_stds[1].set_xticks(P_VALS)
axes_stds[1].set_yticks(Q_VALS)
axes_stds[1].tick_params(labelsize=9)
cbar2s = fig_stds.colorbar(im2s, ax=axes_stds[1])
cbar2s.set_label(r'Std dev', fontsize=10)

filepath_stds_pdf = os.path.join(run_folder, 'fixedpt_vs_nonlinear_stds.pdf')
filepath_stds_pgf = os.path.join(run_folder, 'fixedpt_vs_nonlinear_stds.pgf')
fig_stds.savefig(filepath_stds_pdf, bbox_inches='tight')
fig_stds.savefig(filepath_stds_pgf, bbox_inches='tight')

# --- Figure 3: Fractions ---
fixedpt_frac = fixedpt_mean_heatmap / ANALYZE_MATRIX_SIZE
nonlinear_frac = nonlinear_mean_heatmap / ANALYZE_MATRIX_SIZE

fig_fracs, axes_fracs = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)
fig_fracs.suptitle(r'\Large Fraction: Fixed Point vs Nonlinear (N=' + f'{ANALYZE_MATRIX_SIZE})', 
                   fontsize=14, y=0.97)

im1f = axes_fracs[0].imshow(fixedpt_frac, origin='lower',
                             extent=extent, aspect='auto',
                             cmap='BuGn', interpolation='nearest',
                             vmin=0, vmax=1)
axes_fracs[0].set_title(r'Fixed Point Neurons', fontsize=10)
axes_fracs[0].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_fracs[0].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_fracs[0].set_xticks(P_VALS)
axes_fracs[0].set_yticks(Q_VALS)
axes_fracs[0].tick_params(labelsize=9)
cbar1f = fig_fracs.colorbar(im1f, ax=axes_fracs[0])
cbar1f.set_label(r'Fraction', fontsize=10)
contours1f = axes_fracs[0].contour(P_VALS, Q_VALS, fixedpt_frac,
                                    levels=np.linspace(0, 1, 11),
                                    colors='black', alpha=0.4, linewidths=0.8)
axes_fracs[0].clabel(contours1f, inline=True, fontsize=7, fmt='%.1f')

im2f = axes_fracs[1].imshow(nonlinear_frac, origin='lower',
                             extent=extent, aspect='auto',
                             cmap='RdYlGn_r', interpolation='nearest',
                             vmin=0, vmax=1)
axes_fracs[1].set_title(r'Nonlinear Neurons', fontsize=10)
axes_fracs[1].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_fracs[1].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_fracs[1].set_xticks(P_VALS)
axes_fracs[1].set_yticks(Q_VALS)
axes_fracs[1].tick_params(labelsize=9)
cbar2f = fig_fracs.colorbar(im2f, ax=axes_fracs[1])
cbar2f.set_label(r'Fraction', fontsize=10)
contours2f = axes_fracs[1].contour(P_VALS, Q_VALS, nonlinear_frac,
                                    levels=np.linspace(0, 1, 11),
                                    colors='black', alpha=0.4, linewidths=0.8)
axes_fracs[1].clabel(contours2f, inline=True, fontsize=7, fmt='%.1f')

filepath_fracs_pdf = os.path.join(run_folder, 'fixedpt_vs_nonlinear_fractions.pdf')
filepath_fracs_pgf = os.path.join(run_folder, 'fixedpt_vs_nonlinear_fractions.pgf')
fig_fracs.savefig(filepath_fracs_pdf, bbox_inches='tight')
fig_fracs.savefig(filepath_fracs_pgf, bbox_inches='tight')

# --- Figure 4: Difference and ratio analysis ---
fig_diff, axes_diff = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)
fig_diff.suptitle(r'\Large Comparative Analysis (N=' + f'{ANALYZE_MATRIX_SIZE})', 
                  fontsize=14, y=0.97)

difference_heatmap = fixedpt_mean_heatmap - nonlinear_mean_heatmap
vmax_diff = np.max(np.abs(difference_heatmap))

im1d = axes_diff[0].imshow(difference_heatmap, origin='lower',
                            extent=extent, aspect='auto',
                            cmap='RdBu_r', interpolation='nearest',
                            vmin=-vmax_diff, vmax=vmax_diff)
axes_diff[0].set_title(r'Difference (Fixed Pt - Nonlinear)', fontsize=10)
axes_diff[0].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_diff[0].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_diff[0].set_xticks(P_VALS)
axes_diff[0].set_yticks(Q_VALS)
axes_diff[0].tick_params(labelsize=9)
cbar1d = fig_diff.colorbar(im1d, ax=axes_diff[0])
cbar1d.set_label(r'Count difference', fontsize=10)
contours1d = axes_diff[0].contour(P_VALS, Q_VALS, difference_heatmap,
                                   levels=[0], colors='black', linewidths=1.5)

denominator = fixedpt_mean_heatmap + nonlinear_mean_heatmap
ratio_heatmap = np.zeros_like(fixedpt_mean_heatmap)
mask = denominator > 0.1
ratio_heatmap[mask] = fixedpt_mean_heatmap[mask] / denominator[mask]

im2d = axes_diff[1].imshow(ratio_heatmap, origin='lower',
                            extent=extent, aspect='auto',
                            cmap='PuOr', interpolation='nearest',
                            vmin=0, vmax=1)
axes_diff[1].set_title(r'Ratio: Fixed Pt / (Fixed Pt + Nonlinear)', fontsize=10)
axes_diff[1].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_diff[1].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_diff[1].set_xticks(P_VALS)
axes_diff[1].set_yticks(Q_VALS)
axes_diff[1].tick_params(labelsize=9)
cbar2d = fig_diff.colorbar(im2d, ax=axes_diff[1])
cbar2d.set_label(r'Ratio', fontsize=10)
contours2d = axes_diff[1].contour(P_VALS, Q_VALS, ratio_heatmap,
                                   levels=[0.5], colors='black', linewidths=1.5)

filepath_diff_pdf = os.path.join(run_folder, 'fixedpt_vs_nonlinear_comparison.pdf')
filepath_diff_pgf = os.path.join(run_folder, 'fixedpt_vs_nonlinear_comparison.pgf')
fig_diff.savefig(filepath_diff_pdf, bbox_inches='tight')
fig_diff.savefig(filepath_diff_pgf, bbox_inches='tight')

print("\n" + "="*70)
print("BLOCK 2 SUMMARY: FIXED POINT VS NONLINEAR")
print("="*70)
print(f"Fixed Point Threshold: |dx/dt| < {DERIVATIVE_THRESHOLD}")
print(f"Nonlinear Threshold: Wx+1 < {NONLINEAR_THRESHOLD}\n")

print("FIXED POINT NEURONS:")
print(f"  Mean: {np.mean(fixedpt_mean_heatmap):.2f} ± {np.std(fixedpt_mean_heatmap):.2f}")
print(f"  Fraction: {np.mean(fixedpt_frac):.3f} ± {np.std(fixedpt_frac):.3f}")
print(f"  Range: [{np.min(fixedpt_mean_heatmap):.1f}, {np.max(fixedpt_mean_heatmap):.1f}]\n")

print("NONLINEAR NEURONS:")
print(f"  Mean: {np.mean(nonlinear_mean_heatmap):.2f} ± {np.std(nonlinear_mean_heatmap):.2f}")
print(f"  Fraction: {np.mean(nonlinear_frac):.3f} ± {np.std(nonlinear_frac):.3f}")
print(f"  Range: [{np.min(nonlinear_mean_heatmap):.1f}, {np.max(nonlinear_mean_heatmap):.1f}]\n")

print("COMPARATIVE METRICS:")
print(f"  Mean difference (Fixed Pt - Nonlinear): {np.mean(difference_heatmap):.2f}")
print(f"  Mean ratio (Fixed Pt / Total): {np.mean(ratio_heatmap[mask]):.3f}")
print(f"\nFigures saved to: {run_folder}/")
print(f"  - fixedpt_vs_nonlinear_means.pdf/pgf")
print(f"  - fixedpt_vs_nonlinear_stds.pdf/pgf")
print(f"  - fixedpt_vs_nonlinear_fractions.pdf/pgf")
print(f"  - fixedpt_vs_nonlinear_comparison.pdf/pgf")
print("="*70 + "\n")